## 1. Import the data located at this link. It has information on people infected with dengue at the district level for 2015 to 2021.

In [ ]:
import pandas as pd
# Ruta del archivo CSV: 
ruta_csv = "../../_data/data_dengue_peru.csv" #Ponemos el ../.. para retroceder 

# Cargar el archivo CSV en un DataFrame
dengue_df = pd.read_csv(ruta_csv, low_memory = False)
dengue_df

## 2. Generate ubigeo for Departments and Provinces taking the first two and four numbers. Hint: Use this code.

In [ ]:
#Ubigeo is made out of 6 characters. The first two are a code for the Department. The codes for Department range from 01 to 25, but here we see that the dataframe 
#is ommiting the 0 at the beginning, so we fix that
dengue_df['Ubigeo'] = dengue_df['Ubigeo'].astype(str).str.zfill(6)

In [ ]:
dengue_df #We check if everything is alright

In [ ]:
#Now we can generate an Ubigeo for Department (made out by the first 2 digits) and Provincias (made out by the 4 next digits)
# Extraer los códigos 'Ubigeo' departamentales y provinciales
dengue_df['Ubigeo_Departamental'] = dengue_df['Ubigeo'].str[:2]
dengue_df['Ubigeo_Provincial'] = dengue_df['Ubigeo'].str[2:6]
dengue_df

## 3. Use geopandas to plot the number of cases in 2021 by the district using a continuous legend. Do not forget to indicate the color of NA values. Use this shapefile.

In [ ]:
#1. create new environment
#!conda create -n grupo4
#2. Activate new environment
#!conda activate grupo4
#3. Install package
#!conda install -c conda-forge python=3 geopandas

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Filter the data for the year 2021
dengue_2021 = dengue_df[dengue_df['Año'] == 2021]

# Convert the 'Casos' column to numeric
dengue_2021['Casos'] = pd.to_numeric(dengue_2021['Casos'], errors='coerce')
dengue_2021

In [ ]:
import numpy as np

def custom_sum(series):
    # Check if all values are NaN
    if series.isna().all():
        return np.nan
    # Check if all values are zero or if there is a mix of NaN and zero
    elif (series == 0).all() or (series.fillna(0) == 0).all():
        # If there is at least one NaN in the original series, return NaN
        if series.isna().any():
            return np.nan
        else:
            return 0
    # If there is at least one positive value, replace NaN with 0 and sum all the values
    else:
        return series.fillna(0).sum()

# Apply the custom function to sum the cases by district. It sums the number of weekly cases in each district to obtain the cumulative count for the year 2021 in each district
total_cases_by_district = dengue_2021.groupby('Ubigeo')['Casos'].apply(custom_sum).reset_index()

In [ ]:
total_cases_by_district

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import mapping

# Ruta al shapefile de los departamentos
shapefile_path = '../../_data/INEI_LIMITE_DEPARTAMENTAL.shp'

try:
    # Cargar el shapefile de los departamentos
    departamentos = gpd.read_file(shapefile_path) 

    # Mostrar los primeros registros para verificar la carga
    print(departamentos.head())

except FileNotFoundError:
    print(f"Error: El archivo {shapefile_path} no se encuentra.")
except Exception as e:
    print(f"Ocurrió un error: {e}")

In [ ]:
# Perform the merge based on Ubigeo
merged_gdf = distritos_gdf.merge(total_cases_by_district, left_on='UBIGEO', right_on='Ubigeo')

print(merged_gdf.columns)

In [ ]:
merged_gdf

In [ ]:
# Plot the data
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
# Use the 'Cases' column that exists in merged_gdf.
plot = merged_gdf.plot(column='Casos', 
                       ax=ax, 
                       legend=False,  # Do not create a legend automatically
                       cmap='OrRd',  # Colors for data ranges
                       missing_kwds={'color': 'lightgrey'})  # Color for missing data

# Now the colorbar can be created using the color collection from 'plot', 
#which is the first element in the 'ax' collections list
cbar = fig.colorbar(plot.collections[0], ax=ax)
cbar.set_label('Número de casos de Dengue por distrito en el 2021')
plt.show()

## 4. Aggregate to province level
## Disolve to province level

In [ ]:
prov_shp = distritos_gdf.dissolve( by = 'IDPROV' )
prov_shp

## Aggregate to province level dengue data

In [ ]:
# Apply custom sum group by Ubigeo_Provincial
total_cases_by_province = dengue_2021.groupby('Ubigeo_Provincial')['Casos'].sum().reset_index()
total_cases_by_province

## Merge shp with dengue's data

In [ ]:
# Merge data shape_File at province with the dengue data
merged_gdf = prov_shp.merge(total_cases_by_province, left_on='IDPROV', right_on='Ubigeo_Provincial', how = 'left')
merged_gdf

In [ ]:
# Plot the data
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
#prov_shp.plot( ax = ax )
# Use the 'Cases' column that exists in merged_gdf.
plot = merged_gdf.plot(column='Casos', 
                       ax=ax, 
                       legend=False,  # Do not create a legend automatically
                       cmap='OrRd',  # Colors for data ranges
                       linestyle='--',
                       edgecolor='black', # Black dashed lines 
                       missing_kwds={'color': 'lightgrey'})  # Color for missing data

# Now the colorbar can be created using the color collection from 'plot', 
#which is the first element in the 'ax' collections list
cbar = fig.colorbar(plot.collections[0], ax=ax)
cbar.set_label('Número de casos de Dengue por provincia en el 2021')
plt.show()

## 5. Use geopandas to plot the number of cases by the department for all the years using subplots. Every subplot for each year. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level.

In [ ]:
# Filtramos los datos por la columna "Año" 
# para seleccionar los años 2015 al 2021
casos_dengue_15_21 = dengue_df[(dengue_df["Año"] >= 2015) & (dengue_df["Año"] <= 2021)]

In [ ]:
# Eliminamos las comas de la columna "Casos" y luego convertimos a float
casos_dengue_15_21["Casos"] = casos_dengue_15_21["Casos"].str.replace(',', '').astype(float)

In [ ]:
# Agrupamos y sumamos los casos por departamento y año para los años 2015 al 2021
casos_dengue_departamento_15_21 = casos_dengue_15_21.groupby(["Departamento", "Año"]).agg({"Casos": "sum"}).reset_index()

In [ ]:
# Leemos el archivo shapefile y aplicamos la función
# Creamos un nuevo DataFrame llamado "departamento_shp"
departamento_shp = gpd.read_file(r'../../_data/INEI_LIMITE_DEPARTAMENTAL')
departamento_shp

In [ ]:
# Modificamos el nombre de la columna "NOMBDEP" a "Departamento"
departamento_shp = departamento_shp.rename(columns={'NOMBDEP': 'Departamento'})

In [ ]:
# Unimos las bases de datos utilizando la columna "Departamento"
casos_dengue_departamento_15_21 = casos_dengue_departamento_15_21.merge(departamento_shp, how="right", on="Departamento")
casos_dengue_departamento_15_21

In [ ]:
# Seleccionamos las columnas Departamento, Casos, geometry del DataFrame casos_dengue_departamento_15_21
casos_dengue_departamento_15_21 = casos_dengue_departamento_15_21[['Departamento', 'Casos', 'geometry','Año']]

In [ ]:
from geopandas import GeoDataFrame
# Aplicamos la función GeoDataFrame al DataFrame
casos_dengue_departamento_15_21_gdf = GeoDataFrame(casos_dengue_departamento_15_21)

In [ ]:
# Iteramos sobre los años de interés
for year in range(2015, 2022):
    # Filtramos los datos para el año actual
    casos_dengue_departamento_year = casos_dengue_departamento_15_21[casos_dengue_departamento_15_21["Año"] == year]
    
    # Creamos el GeoDataFrame para el año actual
    casos_dengue_departamento_year_gdf = GeoDataFrame(casos_dengue_departamento_year)
    
    # Hacemos el gráfico
    casos_dengue_departamento_year_gdf.plot(column='Casos', cmap='Purples',
                                            figsize=(20, 20),
                                            linestyle='-',
                                            edgecolor='black',
                                            legend=True,
                                            missing_kwds=dict(color="#E5E5E5"))
    
    # Título
    plt.title(f"Casos de Dengue {year} por Departamento", fontsize=20)
    
    # Mostramos el gráfico
    plt.show()

## 6. Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level. Hint: Use Semana variable to group by quarters.

In [ ]:
# We already have dengue_2021, now we need to convert the weeks to quarters
dengue_2021['Quarter'] = np.select(
    [
        (dengue_2021['Semana'] >= 1) & (dengue_2021['Semana'] <= 13),
        (dengue_2021['Semana'] >= 14) & (dengue_2021['Semana'] <= 26),
        (dengue_2021['Semana'] >= 27) & (dengue_2021['Semana'] <= 39),
        (dengue_2021['Semana'] >= 40) & (dengue_2021['Semana'] <= 53)
    ],
    ['Q1', 'Q2', 'Q3', 'Q4'],
    default='Invalid'
)

dengue_2021

In [ ]:
dengue_2021['Quarter'].unique()

In [ ]:
# Agrupamos y sumamos los casos por departamento y quarters
casos_dengue_departamento_quarters = dengue_2021.groupby(["Departamento", "Quarter"]).agg({"Casos": "sum"}).reset_index()
casos_dengue_departamento_quarters

In [ ]:
# Leemos el archivo shapefile y creamos un nuevo DataFrame llamado "departamento_shp"
departamento_shp = gpd.read_file(r'../../_data/INEI_LIMITE_DEPARTAMENTAL')
departamento_shp

In [ ]:
# Modificamos el nombre de la columna "NOMBDEP" a "Departamento"
departamento_shp = departamento_shp.rename(columns={'NOMBDEP': 'Departamento'})

# Unimos las bases de datos utilizando la columna "Departamento"
casos_dengue_departamento_quarters = casos_dengue_departamento_quarters.merge(departamento_shp, how="right", on="Departamento")
casos_dengue_departamento_quarters

In [ ]:
casos_dengue_departamento_quarters['Departamento']

In [ ]:
# Seleccionamos las columnas Departamento, Casos, geometry del DataFrame casos_dengue_departamento_15_21
casos_dengue_departamento_quarters = casos_dengue_departamento_quarters[['Departamento', 'Casos', 'geometry','Quarter']]
casos_dengue_departamento_quarters


In [ ]:
from geopandas import GeoDataFrame

# Aplicamos la función GeoDataFrame al DataFrame
casos_dengue_departamento_15_21_gdf = GeoDataFrame(casos_dengue_departamento_15_21)


In [ ]:
# !pip install mapclassify

# Iteramos sobre los años de interés
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    # Filtramos los datos para el quarter actual
    casos_dengue_departamento_q = casos_dengue_departamento_quarters[casos_dengue_departamento_quarters["Quarter"] == q]
    
    # Creamos el GeoDataFrame para el quarter actual
    casos_dengue_departamento_q_gdf = GeoDataFrame(casos_dengue_departamento_q)
    
    # Hacemos el gráfico
    casos_dengue_departamento_q_gdf.plot(column='Casos', cmap='Reds',
                                         figsize=(20, 20),
                                         linestyle='-',
                                         edgecolor='black',
                                         legend=True,
                                         missing_kwds=dict(color="#E5E5E5"),
                                         classification_kwds=dict(bins=[20, 30, 40, 50, 100]),
                                         legend_kwds=dict(loc='upper left',
                                                          bbox_to_anchor=(1.01, 1),
                                                          fontsize='x-large',
                                                          title="Casos de Dengue",
                                                          title_fontsize='x-large',
                                                          frameon=False))
    
    # Título
    plt.title(f"Casos de Dengue por Departamento en el {q}", fontsize=20)
    
    # Mostramos el gráfico
    plt.show()
   
    

In [ ]:
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt

# Iteramos sobre los quarters de interés
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    # Filtramos los datos para el quarter actual
    casos_dengue_departamento_q = casos_dengue_departamento_quarters[casos_dengue_departamento_quarters["Quarter"] == q]
    
    # Creamos el GeoDataFrame para el quarter actual
    casos_dengue_departamento_q_gdf = GeoDataFrame(casos_dengue_departamento_q)
    
    # Hacemos el gráfico
    ax = casos_dengue_departamento_q_gdf.plot(column='Casos', cmap='Reds',
                                              figsize=(20, 20),
                                              linestyle='-',
                                              edgecolor='black',
                                              legend=True,
                                              missing_kwds=dict(color="#E5E5E5"),
                                              classification_kwds=dict(bins=[20, 30, 40, 50, 100]))
    # Título
    plt.title(f"Casos de Dengue por Departamento en el {q}", fontsize=20)
    
    # Mostramos el gráfico
    plt.show()


    
    